# CUAD transformer launcher

Select a **GPU runtime**, then run the cells in order. This notebook delegates
chunking, labeling, and training to the repository scripts.

**Before cloning:** commit and push the implementation to the branch configured
below, or upload/extract a complete checkout to `REPO_DIR`. This notebook cannot
access uncommitted files on your computer. Private repositories require access
configured in Colab; do not put credentials in this notebook.

The default run trains pretrained BERT for one epoch on 16 contracts and validates
on four other contracts, using saved fold 0. This is a pipeline smoke test, not a
baseline comparison. Set `MAX_CONTRACTS = None` for the full training split.
The official test split is never used. See `docs/transformer-training.md`.

Setup uses an isolated, locked uv environment (including its own PyTorch).
The GPU check tests that environment, independently of Colab's notebook kernel.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1T-contract-review-challenge.git"
REPO_REF = "main"  # Change to the branch containing the implementation.
REPO_DIR = Path("/content/contract-review")
MAX_CONTRACTS = 20  # None uses all 408 training contracts, split by saved folds.
MAX_STEPS = None  # Optional cap on optimizer steps; None runs complete epochs.
EPOCHS = 1
BATCH_SIZE = 2
VALIDATION_FOLD = 0
SEED = 42
RUN_NAME = "colab-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")


## 1. Checkout and locked environment
An existing checkout is used as-is; this cell does not pull or overwrite it. Initial installation and model downloads may take several minutes.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

required = [
    "pyproject.toml", "uv.lock", "src/train.py", "src/model.py",
    "data/train_separate_questions.json", "docs/tuned-baseline-results/folds.csv",
]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Checkout is missing {missing}. Push the implementation or upload a complete checkout."
    )
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(
    UV + ["sync", "--locked", "--dev", "--python", "3.12"],
    cwd=REPO_DIR, check=True,
)

def run_project(*args):
    return subprocess.run(
        UV + ["run", "--locked", "--no-sync", *map(str, args)],
        cwd=REPO_DIR, check=True,
    )


## 2. GPU and software checks
If the locked environment cannot run a CUDA operation, stop here. Check GPU allocation and the reported driver/runtime compatibility before training.

In [ ]:
run_project("python", "-c", '''
import torch, transformers, accelerate
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("Transformers:", transformers.__version__, "Accelerate:", accelerate.__version__)
assert torch.cuda.is_available(), "No usable CUDA device in the project environment"
print("GPU:", torch.cuda.get_device_name(0))
x = torch.ones(2, device="cuda", requires_grad=True)
x.square().sum().backward()
torch.cuda.synchronize()
assert torch.equal(x.grad.cpu(), torch.tensor([2.0, 2.0]))
print("GPU tensor and gradient check: PASS")
''')
run_project("python", "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider")


## 3. Generate and validate training data
These commands regenerate the training chunks and labels from the committed CUAD source. Each script validates its output. Training checks saved fold assignments before selecting contracts.

In [ ]:
run_project("python", "-B", "-m", "scripts.export_chunks", "--split", "train")
run_project("python", "-B", "-m", "scripts.label_chunks", "--split", "train")


## 4. Train pretrained BERT
The output directory must be new or empty. Rerun the configuration cell to create a new run name. Lower `BATCH_SIZE` if GPU memory is insufficient.

In [ ]:
OUTPUT_DIR = REPO_DIR / "data" / "interim" / "models" / RUN_NAME
args = [
    "python", "-B", "-m", "src.train",
    "--device", "cuda",
    "--epochs", EPOCHS,
    "--batch-size", BATCH_SIZE,
    "--validation-fold", VALIDATION_FOLD,
    "--seed", SEED,
    "--output-dir", OUTPUT_DIR,
]
if MAX_CONTRACTS is not None:
    args += ["--max-contracts", MAX_CONTRACTS]
if MAX_STEPS is not None:
    args += ["--max-steps", MAX_STEPS]
run_project(*args)


## 5. Inspect and download results
The archive contains the checkpoint, tokenizer, validation metrics and predictions, selected contracts, and run metadata. Save it before the runtime is discarded. Smoke metrics do not establish model quality.

In [ ]:
import json

metadata = json.loads((OUTPUT_DIR / "metadata.json").read_text())
assert metadata["checkpoint_reload_verified"]
print(json.dumps(metadata, indent=2))
print((OUTPUT_DIR / "validation_metrics.csv").read_text())


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    str(REPO_DIR.parent / RUN_NAME), "zip",
    root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name,
)
files.download(archive)
